# Match metadata: runtime and operations notes

Operator guide for local execution, tuning, and recovery.


## Standard end-to-end local pipeline sequence

Run from repository root with PDIR set. This sequence runs the full system in order: RR DATA ingest, RR ALLSHEETS ingest, daily transcriptions ingest, transcription QC, metadata matching, monthly QC, regional QC, secondary QC, then SEF export.

```bash
# Optional cleanup for a fresh run of match outputs
scripts/local/clean_match_metadata_outputs.sh --apply --yes

# 1) RR DATA ingest (combined station CSVs under Rainfall-Rescue/DATA)
python scripts/build_rainfall_rescue_parquet.py

# 2) RR ALLSHEETS ingest (individual source-sheet CSVs under Rainfall-Rescue/ALLSHEETS)
python scripts/build_allsheets_parquet.py

# 3) Daily transcriptions ingest
scripts/slurm/submit_ensemble_ingest.sh

# 4) Daily transcriptions QC
scripts/slurm/submit_transcription_qc.sh

# 5) Match metadata (similarity + allsheets + finalise, publishes run manifest)
scripts/slurm/submit_all.sh && sbatch scripts/slurm/assign_metadata.sbatch && scripts/slurm/submit_allsheets.sh

# 6) QC monthly total (QC1)
scripts/slurm/submit_qc.sh

# 7) QC regional stats (QC2 stage 1)
scripts/slurm/submit_daily_consensus.sh
scripts/slurm/submit_regional_stats.sh

# 8) QC secondary (QC2 stage 2 model train + score)
scripts/slurm/submit_secondary_qc.sh

# 9) Export SEF
scripts/slurm/submit_sef_export.sh

# 9) Build database from SEF for fast checks.
scripts/slurm/submit_sef_analysis.sh
```

Notes:
- RR DATA and RR ALLSHEETS ingests are separate steps.
- match_metadata requires the ALLSHEETS parquet dataset to exist.

## Monitoring

```bash
ls -t $PDIR/slurm_logs | head
cat $PDIR/monthly_similarity_parquet/run_manifest/current.json
python scripts/local/verify_match_metadata_manifest.py --comparison-root "$PDIR/monthly_similarity_parquet"
```

Check shard directories and session outputs if a stage fails before rerunning.


## Single-command wrapper (optional)

If you want to run the entire sequence from one terminal command, use this chained invocation:

```bash
python scripts/build_rainfall_rescue_parquet.py && \
python scripts/build_allsheets_parquet.py && \
scripts/slurm/submit_ensemble_ingest.sh && \
scripts/slurm/submit_transcription_qc.sh && \
scripts/slurm/submit_all.sh && sbatch scripts/slurm/assign_metadata.sbatch && scripts/slurm/submit_allsheets.sh && \
scripts/slurm/submit_qc.sh && \
scripts/slurm/submit_regional_stats.sh && \
scripts/slurm/submit_secondary_qc.sh && \
scripts/slurm/submit_sef_export.sh
```

This stops on first failure, so you can inspect logs and rerun from the failed stage.

To resume after a failure, start again from the failed stage command and continue the remaining sequence.

## Tuning knobs

Primary local knobs are configured in scripts/slurm/config.sh and scripts/slurm/config.sh:

- SLURM_QOS
- LOCAL_TOTAL_CORES
- LOCAL_TOTAL_MEM_MB
- NUM_SHARDS and ALLSHEETS_NUM_SHARDS
- TOP_K, MIN_OVERLAP, UNCERTAINTY_WEIGHT, BATCH_SIZE
